1. Из ноутбуков по практике "Рекуррентные и одномерные сверточные нейронные сети" выберите лучшую сеть, либо создайте свою.
2. Запустите раздел "Подготовка"
3. Подготовьте датасет с параметрами `VOCAB_SIZE=20'000`, `WIN_SIZE=1000`, `WIN_HOP=100`, как в ноутбуке занятия, и обучите выбранную сеть. Параметры обучения можно взять из практического занятия. Для  всех обучаемых сетей в данной работе они должны быть одни и теже.
4. Поменяйте размер словаря tokenaizera (`VOCAB_SIZE`) на `5000`, `10000`, `40000`.  Пересоздайте датасеты, при этом оставьте `WIN_SIZE=1000`, `WIN_HOP=100`.
Обучите выбранную нейронку на этих датасетах.  Сделайте выводы об  изменении  точности распознавания авторов текстов. Результаты сведите в таблицу
5. Поменяйте длину отрезка текста и шаг окна разбиения текста на векторы  (`WIN_SIZE`, `WIN_HOP`) используя значения (`500`,`50`) и (`2000`,`200`). Пересоздайте датасеты, при этом оставьте `VOCAB_SIZE=20000`. Обучите выбранную нейронку на этих датасетах. Сделайте выводы об  изменении точности распознавания авторов текстов.

Результаты всей работы сведите в таблицу.

## Подготовка

## 1. Импорт библиотек

In [1]:
# Работа с массивами данных
import numpy as np

# Функции-утилиты для работы с категориальными данными
from tensorflow.keras import utils

# Класс для конструирования последовательной модели нейронной сети
from tensorflow.keras.models import Sequential

# Основные слои
from tensorflow.keras.layers import Dense, Dropout, SpatialDropout1D, BatchNormalization, Embedding, Flatten, Activation
from tensorflow.keras.layers import SimpleRNN, GRU, LSTM, Bidirectional, Conv1D, MaxPooling1D, GlobalMaxPooling1D

# Токенизатор для преобразование текстов в последовательности
from tensorflow.keras.preprocessing.text import Tokenizer

# Рисование схемы модели
from tensorflow.keras.utils import plot_model

# Матрица ошибок классификатора
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Загрузка датасетов из облака google
import gdown

# Функции операционной системы
import os

# Работа со временем
import time

# Регулярные выражения
import re

# Отрисовка графиков
import matplotlib.pyplot as plt

# Вывод объектов в ячейке colab
from IPython.display import display

%matplotlib inline

## 2. Скачивание датасета

In [2]:
# Загрузим датасет из облака
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l7/writers.zip', None, quiet=True)

'writers.zip'

## 3. Распаковка архива

In [3]:
# Распакуем архив в папку writers
!unzip -o writers.zip -d writers/


Archive:  writers.zip
  inflating: writers/(Клиффорд_Саймак) Обучающая_5 вместе.txt  
  inflating: writers/(Клиффорд_Саймак) Тестовая_2 вместе.txt  
  inflating: writers/(Макс Фрай) Обучающая_5 вместе.txt  
  inflating: writers/(Макс Фрай) Тестовая_2 вместе.txt  
  inflating: writers/(О. Генри) Обучающая_50 вместе.txt  
  inflating: writers/(О. Генри) Тестовая_20 вместе.txt  
  inflating: writers/(Рэй Брэдберри) Обучающая_22 вместе.txt  
  inflating: writers/(Рэй Брэдберри) Тестовая_8 вместе.txt  
  inflating: writers/(Стругацкие) Обучающая_5 вместе.txt  
  inflating: writers/(Стругацкие) Тестовая_2 вместе.txt  
  inflating: writers/(Булгаков) Обучающая_5 вместе.txt  
  inflating: writers/(Булгаков) Тестовая_2 вместе.txt  


## 4. Определение констант для загрузки данных (FILE_DIR, SIG_TRAIN, SIG_TEST)

In [4]:
# Настройка констант для загрузки данных
FILE_DIR  = 'writers'                     # Папка с текстовыми файлами
SIG_TRAIN = 'обучающая'                   # Признак обучающей выборки в имени файла
SIG_TEST  = 'тестовая'                    # Признак тестовой выборки в имени файла

## 5. Загрузка и разбор файлов (создание CLASS_LIST, text_train, text_test)

In [5]:
# Подготовим пустые списки

CLASS_LIST = []  # Список классов
text_train = []  # Список для оучающей выборки
text_test = []   # Список для тестовой выборки

# Получим списка файлов в папке
file_list = os.listdir(FILE_DIR)

for file_name in file_list:
    # Выделяем имя класса и типа выборки из имени файла
    m = re.match('\((.+)\) (\S+)_', file_name)
    # Если выделение получилось, то файл обрабатываем
    if m:

        # Получим имя класса
        class_name = m[1]

        # Получим имя выборки
        subset_name = m[2].lower()

        # Проверим тип выборки
        is_train = SIG_TRAIN in subset_name
        is_test = SIG_TEST in subset_name

        # Если тип выборки обучающая либо тестовая - файл обрабатываем
        if is_train or is_test:

            # Добавляем новый класс, если его еще нет в списке
            if class_name not in CLASS_LIST:
                print(f'Добавление класса "{class_name}"')
                CLASS_LIST.append(class_name)

                # Инициализируем соответствующих классу строки текста
                text_train.append('')
                text_test.append('')

            # Найдем индекс класса для добавления содержимого файла в выборку
            cls = CLASS_LIST.index(class_name)
            print(f'Добавление файла "{file_name}" в класс "{CLASS_LIST[cls]}", {subset_name} выборка.')

            # Откроем файл на чтение
            with open(f'{FILE_DIR}/{file_name}', 'r') as f:

                # Загрузим содержимого файла в строку
                text = f.read()
            # Определим выборку, куда будет добавлено содержимое
            subset = text_train if is_train else text_test

            # Добавим текста к соответствующей выборке класса. Концы строк заменяются на пробел
            subset[cls] += ' ' + text.replace('\n', ' ')

Добавление класса "Макс Фрай"
Добавление файла "(Макс Фрай) Тестовая_2 вместе.txt" в класс "Макс Фрай", тестовая выборка.
Добавление класса "О. Генри"
Добавление файла "(О. Генри) Тестовая_20 вместе.txt" в класс "О. Генри", тестовая выборка.
Добавление класса "Рэй Брэдберри"
Добавление файла "(Рэй Брэдберри) Тестовая_8 вместе.txt" в класс "Рэй Брэдберри", тестовая выборка.
Добавление файла "(О. Генри) Обучающая_50 вместе.txt" в класс "О. Генри", обучающая выборка.
Добавление класса "Клиффорд_Саймак"
Добавление файла "(Клиффорд_Саймак) Тестовая_2 вместе.txt" в класс "Клиффорд_Саймак", тестовая выборка.
Добавление файла "(Рэй Брэдберри) Обучающая_22 вместе.txt" в класс "Рэй Брэдберри", обучающая выборка.
Добавление класса "Стругацкие"
Добавление файла "(Стругацкие) Обучающая_5 вместе.txt" в класс "Стругацкие", обучающая выборка.
Добавление файла "(Клиффорд_Саймак) Обучающая_5 вместе.txt" в класс "Клиффорд_Саймак", обучающая выборка.
Добавление класса "Булгаков"
Добавление файла "(Булгако

<>:12: SyntaxWarning: invalid escape sequence '\('
<>:12: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_4632/2214232407.py:12: SyntaxWarning: invalid escape sequence '\('
  m = re.match('\((.+)\) (\S+)_', file_name)


## 6. Определение количества классов CLASS_COUNT

In [6]:
# Определим количество классов
CLASS_COUNT = len(CLASS_LIST)

## 7. Вывод списка классов

In [7]:
# Выведем прочитанные классы текстов
print(CLASS_LIST)

['Макс Фрай', 'О. Генри', 'Рэй Брэдберри', 'Клиффорд_Саймак', 'Стругацкие', 'Булгаков']


## 8. Проверка количества текстов в обучающей выборке

In [8]:
# Посчитаем количество текстов в обучающей выборке
print(len(text_train))

6


## 9. Вывод фрагментов текстов для каждого класса

In [9]:
# Проверим загрузки: выведем начальные отрывки из каждого класса

for cls in range(CLASS_COUNT):                   # Запустим цикл по числу классов
    print(f'Класс: {CLASS_LIST[cls]}')           # Выведем имя класса
    print(f'  train: {text_train[cls][:200]}')   # Выведем фрагмент обучающей выборки
    print(f'  test : {text_test[cls][:200]}')    # Выведем фрагмент тестовой выборки
    print()

Класс: Макс Фрай
  train:  ﻿Власть несбывшегося   – С тех пор как меня угораздило побывать в этой грешной Черхавле, мне ежедневно снится какая-то дичь! – сердито сказал я Джуффину. – Сглазили они меня, что ли? А собственно, по
  test :  ﻿Слишком много кошмаров    Когда балансируешь над пропастью на узкой, скользкой от крови доске, ответ на закономерный вопрос: «Как меня сюда занесло?» – вряд ли принесёт практическую пользу. Зато пои

Класс: О. Генри
  train:  «Лиса-на-рассвете»   Коралио нежился в полуденном зное, как томная красавица в сурово хранимом гареме. Город лежал у самого моря на полоске наносной земли. Он казался брильянтиком, вкрапленным в ярко
  test :  ﻿Багдадская птица   Без всякого сомнения, дух и гений калифа Гаруна аль-Рашида осенил маркграфа Августа-Михаила фон Паульсена Квигга.  Ресторан Квигга находится на Четвертой авеню — на улице, которую

Класс: Рэй Брэдберри
  train:  ﻿451° по Фаренгейту   ДОНУ КОНГДОНУ С БЛАГОДАРНОСТЬЮ   Если тебе дадут линованную бумагу, пиши

## 10. Определение класса timex для измерения времени операций

In [10]:
# Контекстный менеджер для измерения времени операций
# Операция обертывается менеджером с помощью оператора with

class timex:
    def __enter__(self):
        # Фиксация времени старта процесса
        self.t = time.time()
        return self

    def __exit__(self, type, value, traceback):
        # Вывод времени работы
        print('Время обработки: {:.2f} с'.format(time.time() - self.t))

## Решение

## 12. Определение базовых гиперпараметров

In [11]:
# БАЗОВЫЕ параметры (как в задании)
VOCAB_SIZE = 20000
WIN_SIZE = 1000
WIN_HOP = 100

BATCH_SIZE = 64
EPOCHS = 10

## 13. Создание токенизатора и преобразование текстов в последовательности

In [12]:
# Создаём токенизатор
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='UNK')
tokenizer.fit_on_texts(text_train)

# Преобразуем тексты в последовательности
seq_train = tokenizer.texts_to_sequences(text_train)
seq_test = tokenizer.texts_to_sequences(text_test)

## 14. Функция create_dataset для нарезки окон


In [13]:
def create_dataset(seq_list, win_size, win_hop):
    X = []
    y = []

    for class_id, seq in enumerate(seq_list):
        for i in range(0, len(seq) - win_size, win_hop):
            X.append(seq[i:i + win_size])
            y.append(class_id)

    return np.array(X), utils.to_categorical(y, CLASS_COUNT)

## 15. Создание обучающей и тестовой выборок

In [14]:
with timex():
    X_train, y_train = create_dataset(seq_train, WIN_SIZE, WIN_HOP)
    X_test, y_test = create_dataset(seq_test, WIN_SIZE, WIN_HOP)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

Время обработки: 3.22 с
(18417, 1000) (18417, 6)
(6968, 1000) (6968, 6)


## 16. Модель

In [15]:
def create_model():
    model = Sequential()

    model.add(Embedding(VOCAB_SIZE, 128, input_length=WIN_SIZE))
    model.add(SpatialDropout1D(0.2))

    model.add(Conv1D(128, 5, activation='relu'))
    model.add(MaxPooling1D(5))

    model.add(Bidirectional(LSTM(64, return_sequences=False)))

    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.3))

    model.add(Dense(CLASS_COUNT, activation='softmax'))

    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model

## 17. Обучение модели с базовыми параметрами

In [16]:
model = create_model()

with timex():
    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 23s 45ms/step - accuracy: 0.7167 - loss: 0.7266 - val_accuracy: 0.6343 - val_loss: 1.3973
Epoch 2/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 11s 40ms/step - accuracy: 0.9778 - loss: 0.0734 - val_accuracy: 0.6751 - val_loss: 1.6565
Epoch 3/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 12s 40ms/step - accuracy: 0.9995 - loss: 0.0044 - val_accuracy: 0.7024 - val_loss: 2.0403
Epoch 4/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 11s 40ms/step - accuracy: 0.9976 - loss: 0.0111 - val_accuracy: 0.7186 - val_loss: 1.7084
Epoch 5/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 12s 40ms/step - accuracy: 1.0000 - loss: 9.6730e-04 - val_accuracy: 0.7160 - val_loss: 1.9674
Epoch 6/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 12s 41ms/step - accuracy: 0.9979 - loss: 0.0079 - val_accuracy: 0.6858 - val_loss: 1.8163
Epoch 7/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 12s 41ms/step - accuracy: 0.9962 - loss: 0.0121 - val_accuracy: 0.6797 - val_loss: 1.5562
Epoch 8/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 12s 43ms/step - accuracy: 0.9990 - loss: 0.004

## 18. Оценка точности на тестовой выборке

In [17]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Accuracy (VOCAB_SIZE=20000): {acc:.4f}')

Accuracy (VOCAB_SIZE=20000): 0.6544


## 19. Эксперимент с разным VOCAB_SIZE (5000, 10000, 40000)

In [20]:
vocab_results = []

for vocab in [5000, 10000, 40000]:
    print(f'\n===== VOCAB_SIZE = {vocab} =====')

    tokenizer = Tokenizer(num_words=vocab, oov_token='UNK')
    tokenizer.fit_on_texts(text_train)

    seq_train = tokenizer.texts_to_sequences(text_train)
    seq_test = tokenizer.texts_to_sequences(text_test)

    X_train, y_train = create_dataset(seq_train, WIN_SIZE, WIN_HOP)
    X_test, y_test = create_dataset(seq_test, WIN_SIZE, WIN_HOP)

    model = create_model()

    model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print(f'Accuracy: {acc:.4f}')

    vocab_results.append((vocab, acc))


===== VOCAB_SIZE = 5000 =====


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Accuracy: 0.6633

===== VOCAB_SIZE = 10000 =====
Accuracy: 0.7137

===== VOCAB_SIZE = 40000 =====
Accuracy: 0.6425


## 20. Эксперимент с разными WIN_SIZE и WIN_HOP [(500,50), (1000,100), (2000,200)]

In [21]:
window_results = []

for win_size, win_hop in [(500, 50), (1000, 100), (2000, 200)]:
    print(f'\n===== WIN_SIZE={win_size}, WIN_HOP={win_hop} =====')

    tokenizer = Tokenizer(num_words=20000, oov_token='UNK')
    tokenizer.fit_on_texts(text_train)

    seq_train = tokenizer.texts_to_sequences(text_train)
    seq_test = tokenizer.texts_to_sequences(text_test)

    X_train, y_train = create_dataset(seq_train, win_size, win_hop)
    X_test, y_test = create_dataset(seq_test, win_size, win_hop)

    model = Sequential()

    model.add(Embedding(20000, 128, input_length=win_size))
    model.add(SpatialDropout1D(0.2))
    model.add(Conv1D(128, 5, activation='relu'))
    model.add(MaxPooling1D(5))
    model.add(Bidirectional(LSTM(64)))
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(CLASS_COUNT, activation='softmax'))

    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print(f'Accuracy: {acc:.4f}')

    window_results.append((win_size, win_hop, acc))


===== WIN_SIZE=500, WIN_HOP=50 =====
Accuracy: 0.6737

===== WIN_SIZE=1000, WIN_HOP=100 =====
Accuracy: 0.6326

===== WIN_SIZE=2000, WIN_HOP=200 =====
Accuracy: 0.6559


## 21. Вывод итоговых результатов экспериментов

In [22]:
print('\n=== VOCAB_SIZE RESULTS ===')
for v, acc in vocab_results:
    print(f'{v}: {acc:.4f}')

print('\n=== WINDOW RESULTS ===')
for w, h, acc in window_results:
    print(f'WIN_SIZE={w}, WIN_HOP={h}: {acc:.4f}')


=== VOCAB_SIZE RESULTS ===
5000: 0.6633
10000: 0.7137
40000: 0.6425

=== WINDOW RESULTS ===
WIN_SIZE=500, WIN_HOP=50: 0.6737
WIN_SIZE=1000, WIN_HOP=100: 0.6326
WIN_SIZE=2000, WIN_HOP=200: 0.6559
